# 08 Trajectory Preprocessing

This notebook prepares GROMACS production trajectories before RMSD, radius of gyration, SASA, or other structural analysis.

Raw periodic trajectories are often not analysis-ready. A polymer can cross the unit-cell boundary, molecules can look split, and the whole system can drift through the box. Those effects can create noisy RMSD/Rg traces that describe box handling rather than molecular behavior.

## External trajectory storage

Molecular dynamics trajectories are usually stored outside Git repositories. Files such as `.xtc`, `.trr`, `.edr`, `.cpt`, and long production logs can become very large, often from hundreds of megabytes to many gigabytes per run. Adding them to Git makes cloning, pulling, pushing, and reviewing the repository slow and can quickly exceed hosting limits.

A practical workflow is to keep code, notebooks, input templates, small reference structures, and metadata in Git, while storing large raw and processed trajectories on external storage such as an external hard drive, institutional storage, or HPC project storage. The notebook can then point to those files with a local path or copied working directory.

For reproducibility, keep a lightweight record in Git that describes where the trajectory came from: system name, simulation engine, `.mdp` files, topology, index file, production date, and storage location. Do not commit large trajectory files unless there is a specific reason and the project is configured for large-file storage.

## Strategy

The workflow uses a dedicated GROMACS index group named `[ center ]`.

For the current polymer-only workflow, `[ center ]` is generated from `[ PHA ]`. The preprocessing commands then select `center` for centering or fitting and `System` for output.

This keeps the full solvated system, including water and ions, while keeping the polymer centered. The command layer never needs to hardcode `PHA`, so future systems can map `[ center ]` to `Protein`, `MEMB | Protein`, or `Protein | PHA` without rewriting preprocessing calls.

## Why full-group centering

Centering on a single atom is fragile for polymers because one atom can sit near a periodic boundary while the rest of the chain spans another image. Centering on the full target group uses the group geometry, giving a more stable representation for visual inspection and downstream trajectory analysis.

In [ ]:
from pathlib import Path

from iphasimulator.trajectory import ensure_center_index, preprocess_gromacs_trajectory, read_index

# All required GROMACS files are in this folder.
SYSTEM_DIR = Path("/Users/k20098771/Data/MD_projects/PHA/MD/gromacs/PHB4_01")

TRAJECTORY_PATH = SYSTEM_DIR / "step7_production.xtc"
STRUCTURE_PATH = SYSTEM_DIR / "step7_production.tpr"
INDEX_PATH = SYSTEM_DIR / "index.ndx"
VISUALIZATION_STRUCTURE_PATH = SYSTEM_DIR / "step6.2_npt.gro"

{
    "system_dir": SYSTEM_DIR,
    "trajectory": TRAJECTORY_PATH,
    "structure_for_preprocessing": STRUCTURE_PATH,
    "index": INDEX_PATH,
    "structure_for_visualization": VISUALIZATION_STRUCTURE_PATH,
}


## Input files

All files for this example are expected in:

`/Users/k20098771/Data/MD_projects/PHA/MD/gromacs/PHB4_01/`

The notebook uses:

- `step7_production.xtc` as the raw production trajectory
- `step7_production.tpr` as the structure/topology input for `gmx trjconv`
- `index.ndx` as the GROMACS index file
- `step6.2_npt.gro` as the structure file for visualization

Processed outputs are written back into the same folder under `processed/` and `analysis_ready/`.


## Step 1: create or reuse `center.ndx`

If `center.ndx` already contains `[ center ]`, it is reused. Otherwise the helper reads `index.ndx`, resolves the workflow mapping, and writes a reusable `center.ndx` with the original groups plus `[ center ]`.

In [ ]:
center_result = ensure_center_index(
    INDEX_PATH,
    SYSTEM_DIR / "center.ndx",
    workflow_type="polymer",
)

center_index = read_index(center_result.index_path)
{
    "center_index": str(center_result.index_path),
    "created": center_result.created,
    "reused_existing_center": center_result.reused_existing_center,
    "source_groups": center_result.source_groups,
    "center_atom_count": len(center_index.group("center")),
}


## Step 2: center, reconstruct molecules, and compact-wrap

The first `trjconv` pass reconstructs molecules across periodic boundaries, uses a compact unit cell, and centers on `[ center ]`:

```bash
echo -e "center\nSystem" | gmx trjconv \
  -f step7_production.xtc \
  -s step7_production.tpr \
  -n center.ndx \
  -pbc mol \
  -ur compact \
  -center \
  -o processed/step7_centered.xtc
```

Selection 1 is the centering group. Selection 2 is the output group. Use `System` for output so water and ions stay in the trajectory.


## Step 3: optional fitting/alignment

The second pass removes overall rotation and translation relative to the reference structure:

```bash
echo -e "center\nSystem" | gmx trjconv \
  -f processed/step7_centered.xtc \
  -s step7_production.tpr \
  -n center.ndx \
  -fit rot+trans \
  -o analysis_ready/step7_fitted.xtc
```

The fitted trajectory is the preferred input for RMSD-like analyses. The centered trajectory is useful when fitting is not appropriate for a specific observable.


In [ ]:
required = [TRAJECTORY_PATH, STRUCTURE_PATH, INDEX_PATH]
have_required_inputs = all(path.exists() for path in required)

outputs = preprocess_gromacs_trajectory(
    SYSTEM_DIR,
    trajectory=TRAJECTORY_PATH,
    structure=STRUCTURE_PATH,
    index=INDEX_PATH,
    workflow_type="polymer",
    fit=True,
    extract_representative_frame=True,
    dry_run=not have_required_inputs,
)

{
    "have_required_inputs": have_required_inputs,
    "raw": str(outputs.raw_trajectory_path),
    "structure": str(outputs.structure_path),
    "visualization_structure": str(VISUALIZATION_STRUCTURE_PATH),
    "centered": str(outputs.centered_trajectory_path),
    "analysis_ready": str(outputs.analysis_trajectory_path),
    "representative_frame": str(outputs.representative_frame_path),
}


If `have_required_inputs` is `False`, check that these files exist in `SYSTEM_DIR`: `step7_production.xtc`, `step7_production.tpr`, and `index.ndx`.


## Before and after checks

The raw trajectory is the direct production output. The centered trajectory should show the polymer as one molecule in a compact solvent box. The fitted trajectory should remove whole-system drift and rotation while preserving solvent and ions in the output group.

In [ ]:
for label, path in {
    "raw": outputs.raw_trajectory_path,
    "centered": outputs.centered_trajectory_path,
    "analysis_ready": outputs.analysis_trajectory_path,
    "representative_frame": outputs.representative_frame_path,
}.items():
    if path is not None:
        print(f"{label:20s} {path.exists()}  {path}")

## Visualize representative frames

For visualization, use `step6.2_npt.gro` as the structure reference and open the processed trajectory from `outputs.analysis_trajectory_path`.


In [ ]:
frame_path = outputs.representative_frame_path
analysis_trajectory_path = outputs.analysis_trajectory_path

if frame_path is None or not frame_path.exists():
    print("Representative frame is not available yet. Check the preprocessing inputs and rerun preprocessing.")
else:
    try:
        from IPython.display import display
        import nglview as nv

        if VISUALIZATION_STRUCTURE_PATH.exists() and analysis_trajectory_path.exists():
            # Load the GRO structure, then add the processed XTC trajectory.
            view = nv.show_file(str(VISUALIZATION_STRUCTURE_PATH))
            view.add_trajectory(str(analysis_trajectory_path))
        else:
            view = nv.show_file(str(frame_path))
        display(view)
    except ImportError:
        print("Representative frame:", frame_path)
        print("Visualization structure:", VISUALIZATION_STRUCTURE_PATH)
        print("Processed trajectory:", analysis_trajectory_path)
